 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

Reading from cache.


In [2]:
PROMPT_END_STR_PERPLEX = '---'    
RESPONSE_SOURCES_DIVIDER_STR = '<div style="text-align: center">⁂</div>'
source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')

perplex_source_list_OLD_FORMATre = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
# sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)') # note used?

relinker = lpz.relinker

def split_single_prompt_response_text_perplex(pr_text: str) -> Tuple[str, str, str, str]:
    """Splits perplexity output markdown text into prompt, response and source sections.
    In the response, duplicate citenums are removed, and the mapping from original 
    to deduplicated numbers is in citenumes_to_url_source"""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', pr_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    preamble = pr_text[:heading_start_index].strip()
    
    prompt_end_index, response_start_index = rfw.find_markdown_divider_boundaries(pr_text)
        
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}. '
                         'Probably missed the starting level 1 header part of the prompt.')
    
    prompt = pr_text[heading_start_index:prompt_end_index+1].strip()

    response_sources_divider_index = pr_text.rfind(RESPONSE_SOURCES_DIVIDER_STR)

    if response_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if response_sources_divider_index <= response_start_index:
        raise ValueError('body_sources_divider_index <= response_sources_divider_index')
    
    response = f"{pr_text[response_start_index:response_sources_divider_index]}".strip()
    
    sources = pr_text[response_sources_divider_index:]
    citenum_url_pairs = rfw.get_link_tu_pairs(sources, source_list_pattern_perplex)
    
    return lpz.PromptResponseSplit(preamble, prompt, response, citenum_url_pairs)

In [3]:
def split_single_prompt_response_dedup_perplex(markdown_text: str) -> lpz.PromptResponseSplitDeDup:
    """Splits perplexity output markdown text into prompt, response and source sections.
    In the response, duplicate citenums are removed, and the mapping from original 
    to deduplicated numbers is in citenumes_to_url_source"""
    
    return relinker.split_prompt_response_dedup(markdown_text, split_single_prompt_response_text_perplex)

def relink_single_file_perplexity(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    "Relinks and writes to a file a single prompt/response from perplexity."
    file_text = lpz.read_markdown_file(perplexity_file)
    
    prsplit = split_single_prompt_response_dedup_perplex(file_text)
    body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit,'plain_link')
    body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)

    relinked_file.write_text(f'{lpz.make_obsidian_front_matter()}\n*{source_link}*\n# Prompt\n\n{prsplit.prompt}\n'
                             f'# Response\n\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', 
                             encoding='utf-8')

In [4]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
# perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / "perple_new_format_longprompt_example.md"
perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex' / 'GPT-4o.md'

output_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Scratch Space")

output_file = output_dir / "tmp_perplex_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_single_file_perplexity(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/GPT-4o.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_perplex_example.md')
Done.


### Test merging

In [5]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

# multi-file, same prompt
#chat_files = list(datdir.glob('*.md'))
# multi-file, different prompt
#chat_files = [chat_files[3], pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_example.md")]
# single file
#chat_files = [chat_files[3]]

# single smc file but multiprompt
chat_files = [pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_multi_prompt_savemychatbot_example.md")]


merged_output_file = output_dir / 'tmp_stock_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [21]:
from click import prompt


def split_all_prs(chat_fle):
    file_text = lpz.read_markdown_file(chat_file)

    if not lpz.is_smc_content(file_text):
        # perplexity files only ever have one pr
        return [split_single_prompt_response_dedup_perplex(file_text)]

    # raise ValueError('SMC not implemented yet')
    #chat_source = rfw.file_link_md('source', chat_file)
    # relinked_sections = [lpz.make_obsidian_front_matter() + f'{chat_source.lstrip()}\n']
    prsplits = []    
    sections = re.split(rf'(?<=\n){lpz.PROMPT_HEADER_SMC}', file_text)
    for section in sections[1:]:  # Process each user section
        section = f'{lpz.PROMPT_HEADER_SMC}\n{section}' # stick header back on for more certtain matching
        prsplits.append(lpz.split_single_prompt_response_dedup_smc(section))

    return prsplits


verbose = True
if verbose:
    print(f'BOB')


num_chat_files = len(chat_files)
all_prompts, all_responses, all_citenums_to_url = [], [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'Parsing {chat_file.stem}')

    for prompt_index, prsplit in enumerate(split_all_prs(chat_file)):
        print(f'{file_index=}, {prompt_index=}')
        all_prompts.append(prsplit.prompt)
        all_responses.append(prsplit.response_dedup)
        
        display(prsplit.citenums_to_url_source)

        citenums_to_url_source = prsplit.citenums_to_url_source.copy()
        citenums_to_url_source[['file_index','chat_file', 'prompt_index']] = file_index, chat_file, prompt_index
        all_citenums_to_url.append(citenums_to_url_source.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} total citation numbers')

Reading from cache.
BOB
Parsing perplexity_multi_prompt_savemychatbot_example
Sources header(\*\*Sources\*\*) not in expected place or no source list: Assume no sources.
https://no_url_in_response_for_bare_cite_number 20
getting citenum_url_pairs from response.  sources=''


[('1', 'https://en.wikipedia.org/wiki/mutual_information'),
 ('5', 'http://www.scholarpedia.org/article/mutual_information'),
 ('1', 'https://en.wikipedia.org/wiki/mutual_information'),
 ('4', 'https://quantdare.com/what-is-mutual-information'),
 ('10', 'https://pmc.ncbi.nlm.nih.gov/articles/pmc10132663'),
 ('3', 'https://www.blog.trainindata.com/mutual-information-with-python'),
 ('18',
  'https://math.stackexchange.com/questions/3020611/how-to-calculate-mutual-information'),
 ('18',
  'https://math.stackexchange.com/questions/3020611/how-to-calculate-mutual-information'),
 ('3', 'https://www.blog.trainindata.com/mutual-information-with-python'),
 ('10', 'https://pmc.ncbi.nlm.nih.gov/articles/pmc10132663'),
 ('6', 'https://www.nature.com/articles/srep10981'),
 ('6', 'https://www.nature.com/articles/srep10981'),
 ('7', 'https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197'),
 ('10', 'https://pmc.ncbi.nlm.nih.gov/articles/pmc10132663'),
 ('10', 'https://pmc.ncbi.nlm.nih.gov/articles/pmc1013

,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/mutual_information
5,2,http://www.scholarpedia.org/article/mutual_inf...
4,3,https://quantdare.com/what-is-mutual-information
12,3,https://quantdare.com/what-is-mutual-information
10,4,https://pmc.ncbi.nlm.nih.gov/articles/pmc10132663
3,5,https://www.blog.trainindata.com/mutual-inform...
18,6,https://math.stackexchange.com/questions/30206...
6,7,https://www.nature.com/articles/srep10981
7,8,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197


Sources header(\*\*Sources\*\*) not in expected place or no source list: Assume no sources.
getting citenum_url_pairs from response.  sources=''


[('1', 'https://libanswers.lib.miamioh.edu/stats-faq/faq/343635'),
 ('3',
  'https://stats.stackexchange.com/questions/81659/mutual-information-versus-correlation'),
 ('3',
  'https://stats.stackexchange.com/questions/81659/mutual-information-versus-correlation'),
 ('17',
  'https://towardsdatascience.com/how-to-measure-relationship-between-variables-d0606df27fd8'),
 ('7', 'https://mattiheino.com/2019/05/10/correlation'),
 ('10', 'https://m-clark.github.io/docs/correlationcomparison.pdf'),
 ('9',
  'https://www.stats.ox.ac.uk/~cucuring/lecture_2_correlations_dependence.pdf'),
 ('10', 'https://m-clark.github.io/docs/correlationcomparison.pdf'),
 ('10', 'https://m-clark.github.io/docs/correlationcomparison.pdf'),
 ('(1) Mutual information',
  'https://en.wikipedia.org/wiki/mutual_information'),
 ('(2) PDF Entropy and Mutual Information',
  'https://people.cs.umass.edu/~elm/teaching/docs/mutinf.pdf'),
 ("(3) Mutual information with Python - Train in Data's Blog",
  'https://www.blog.train

...

file_index=0, prompt_index=1


,new_num,url
orig_num,,
1,1,https://libanswers.lib.miamioh.edu/stats-faq/f...
3,2,https://stats.stackexchange.com/questions/8165...
17,3,https://towardsdatascience.com/how-to-measure-...
7,4,https://mattiheino.com/2019/05/10/correlation
10,5,https://m-clark.github.io/docs/correlationcomp...
9,6,https://www.stats.ox.ac.uk/~cucuring/lecture_2...
(1) Mutual information,7,https://en.wikipedia.org/wiki/mutual_information
(21) Mutual information - Wikipedia,7,https://en.wikipedia.org/wiki/mutual_information
(2) PDF Entropy and Mutual Information,8,https://people.cs.umass.edu/~elm/teaching/docs...


file_index=0, prompt_index=2


,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/total_correlation
4,2,https://stats.stackexchange.com/questions/8165...
5,3,http://www.mathemafrica.org?p=16127
3,4,https://arxiv.org/abs/2011.04794
6,5,https://proceedings.mlr.press/v206/bai23a/bai2...
(1) Mutual information,6,https://en.wikipedia.org/wiki/mutual_information
(21) Mutual information - Wikipedia,6,https://en.wikipedia.org/wiki/mutual_information
(2) PDF Entropy and Mutual Information,7,https://people.cs.umass.edu/~elm/teaching/docs...
(3) Mutual information with Python - Train in Data's Blog,8,https://www.blog.trainindata.com/mutual-inform...


Found 66 total citation numbers


#### Make a unified cite number set for the merged document

In [7]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Double sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['new_num_int'] = df['new_num'].astype(int) # so can sort

grouped = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_new_num_int=('new_num_int', 'mean'),
    mean_prompt_index=('prompt_index', 'mean')
).reset_index()

grouped = grouped.sort_values(by=['mean_file_index', 'mean_prompt_index', 'mean_new_num_int'], 
                              ascending=True).reset_index(drop=True)

grouped['citenum_merged'] = np.arange(1, len(grouped) + 1).astype(str) # citenum == rank as sttring

# Merge back the new citenumes
df = df.merge(grouped[['url', 'citenum_merged']], on='url')

In [8]:
all_citenums_to_url = (df.sort_values('citenum_merged')
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('new_num_int', axis=1)
                       .set_index('file_index'))

#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [9]:
# TODO: warn if more than one prompt in a file (only possible for SMC)
# TODO: will this work for a single file?
all_prompts_same = True
for i in range(0,len(all_prompts)-1):
    is_same = all_prompts[i].strip().lower() == all_prompts[i+1].strip().lower()
    all_prompts_same &= is_same

In [14]:
all_citenums_to_url

,orig_num,doc_dedup_num,url,chat_file,prompt_index,new_num
file_index,,,,,,
0,20,9,https://no_url_in_response_for_bare_cite_numbe...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,0,1
0,7,8,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,0,10
0,(7) A Quick and Easy Way to Estimate Entropy a...,13,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1,10
0,(7) A Quick and Easy Way to Estimate Entropy a...,12,https://pmc.ncbi.nlm.nih.gov/articles/pmc8239197,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,10
0,10,4,https://pmc.ncbi.nlm.nih.gov/articles/pmc10132663,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,0,11
...,...,...,...,...,...,...
0,(3) Mutual information with Python - Train in ...,8,https://www.blog.trainindata.com/mutual-inform...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,8
0,(3) Mutual information with Python - Train in ...,9,https://www.blog.trainindata.com/mutual-inform...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1,8
0,(6) Mutual Information between Discrete Variab...,11,https://www.nature.com/articles/srep10981,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,9


In [11]:
# make a single mapping from unified citenums to urls
unified_citenums = all_citenums_to_url[['new_num', 'url']].drop_duplicates()
unified_citenums.index = unified_citenums['new_num']

is_multi_file_chat = num_chat_files > 1
do_single_top_prompt = all_prompts_same and is_multi_file_chat

pr_ix_nms = ['file_index','prompt_index']
pr_indices = all_citenums_to_url.reset_index()[pr_ix_nms].sort_values(pr_ix_nms).drop_duplicates().reset_index(drop=True)

all_promptresp_unified, chat_source_file_link = '', []
for response_index, (file_index, prompt_index) in pr_indices.iterrows():
# for file_index, response_dedup in enumerate(all_responses):
    if verbose:
        print(f'Unifying {chat_files[file_index].stem}, {prompt_index=}')
        
    # remap deduped citenums to unified citenums
    citenums_to_url_this = all_citenums_to_url.loc[file_index]
    citenums_dedup_to_unified = citenums_to_url_this.set_index('doc_dedup_num').new_num.to_dict()
    response_unified = relinker.replace_response_citenums(all_responses[response_index], citenums_dedup_to_unified) # unified citenums

    # Put this body within the appropriate merged dialog headings
    source_link = rfw.file_link_md('source', chat_files[file_index])
    if is_multi_file_chat:
        if do_single_top_prompt:
            if file_index == 0:
                all_promptresp_unified += f'# Prompt\n\n{all_prompts[file_index]}\n# Responses\n'
            all_promptresp_unified += f'\n## {chat_files[file_index].name}\n*{source_link}*\n\n'
            all_promptresp_unified += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)            
        else:
            short_prompt = rfw.get_first_n_words(all_prompts[file_index], lpz.MAX_WORDS_PROMPT_HEADING)
            all_promptresp_unified += f'\n# {short_prompt}\n*{source_link}*\n\n{all_prompts[file_index]}\n## Response\n\n'
            all_promptresp_unified += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)
    else:
        all_promptresp_unified += f'*{source_link}*\n\n# Prompt\n\n{all_prompts[file_index]}\n# Response\n'
        all_promptresp_unified += response_unified

Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=0
Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=1
Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=2


##### Insert links to Obsidian or Zotero

In [13]:
print(f'writing to {merged_output_file=}')
# get it in the expected structure
prsplit_all = lpz.PromptResponseSplitDeDup('','',all_promptresp_unified, all_citenums_to_url)

all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit_all, 'plain_link')
#all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(all_promptresp_unified , unified_citenums, 'plain_link')
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group('num'))))
merged_output_file.write_text(f'{lpz.make_obsidian_front_matter()}\n{all_promptresp_unified_relinked}\n# Citations\n{relinked_sources}',
                              encoding='utf-8')
print("Done.")

writing to merged_output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_stock_perplexy_merged.md')
Done.
